In [9]:
from apis_core.apis_metainfo.models import Uri, Collection
from csae_pyutils import gsheet_to_df

from normdata.utils import import_from_normdata
from tqdm import tqdm
from time import sleep

In [2]:
col = Collection.objects.filter(name__startswith="ÖBL nach").first()
domain = "oebl"


In [3]:
df = gsheet_to_df("1TnVJu6V4jcRDiL5wJ_GlEG19pr7hZaZcic9B09OswC8")

200


In [4]:
df.head()

,person,personLabel,birthDate,oblId,doi
0,http://www.wikidata.org/entity/Q28470206,Therese Kilanyi,1830-01-01T00:00:00Z,K/Kilanyi_Therese_1830_1881,10.1553/0X00282866
1,http://www.wikidata.org/entity/Q37973154,Heinrich Thurnes,1830-01-01T00:00:00Z,T/Thurnes_Heinrich_1833_1865,10.1553/0X00311149
2,http://www.wikidata.org/entity/Q56007519,Ferdinand Krauthauf,1830-01-01T00:00:00Z,K/Krauthauf_Ferdinand_1830_1893,10.1553/0X002824A8
3,http://www.wikidata.org/entity/Q60623771,Wenzel Hartl,1830-01-01T00:00:00Z,H/Hartl_Wenzel_1830_1895,10.1553/0X00281C61
4,http://www.wikidata.org/entity/Q60818189,Enrico Matcovich,1830-01-01T00:00:00Z,M/Matcovich_Enrico_1830_1898,10.1553/0X00283405


In [5]:
filtered = df[df["doi"].notna()]
print(len(df), len(filtered))

4475 4226


In [ ]:
for wikidata_id, ndf in tqdm(filtered.groupby('person')):
    doi = f"https://doi.org/{ndf.iloc[0]['doi']}"
    new_uri, created = Uri.objects.get_or_create(uri=doi, domain=domain)
    if created:
        sleep(2)
    entity = import_from_normdata(wikidata_id, "person")
    new_uri.entity = entity
    new_uri.save()
    entity.collection.add(col)


  4%|████████                                                                                                                                                                                                               | 151/4028 [01:46<3:51:24,  3.58s/it]